# Lesson 10 · BatchNorm, LayerNorm & RMSNorm: writing the three normalizations yourself and testing them on 10–30-layer MNIST networks

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/hoc-sau-notebooks/blob/main/10-chuan-hoa-en.ipynb)

This notebook accompanies [Lesson 10 of Học Sâu · Reels](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/chuan-hoa/), a deep-learning course originally in
Vietnamese. It is the English edition of the course's notebook
([Vietnamese original](https://github.com/trungnguyenvn/hoc-sau-notebooks/blob/main/10-chuan-hoa.ipynb)):
the code, the data and every check are the same. The interactive demo is on the lesson page.
In Lesson 09, the deep 784 → L×64 → 10 network was hard to train because the signal and the gradient shrink or swell
from layer to layer. Here we insert a **normalization** step into every hidden layer (Linear → normalization → ReLU),
write BatchNorm, LayerNorm and RMSNorm ourselves in numpy, both the forward and the backward pass, then measure on
MNIST how far they help a deep network, and where they don't.

**What you'll do:**
1. Write BatchNorm, LayerNorm and RMSNorm (forward and backward) and check the gradients with finite differences.
2. Insert them into a deep network, measure how the distribution of a hidden layer's inputs drifts during training,
   and try making it drift on purpose.
3. Apply the three methods to the real activations of a trained network: normalizing by column (over the batch) or
   by row (per sample), and what γ, β are for.
4. Compare BatchNorm at training time (the batch's statistics) with inference time (the running mean).
5. Train 10–30-layer networks with each method, read the results of the full sweep, and find out why BatchNorm
   doesn't rescue the 30-layer network.

**Background:** [Lesson 09 · Vanishing gradients & initialization](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/vanishing-gradient/)
(the same deep network, He initialization), [Lesson 06 · Backpropagation](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/lan-truyen-nguoc/)
(the backward pass, reused in section 5) and [Lesson 05 · Gradient descent](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/gradient-descent/)
(SGD, the learning rate $\eta$). **Run time:** under a minute on a CPU, including the first MNIST download (~11 MB).

Every number in the notebook is computed by the cells below, and matches the numbers in the demo. Only the results of
the full sweep (216 training runs) are loaded from the very file the demo uses, because rerunning it takes a few
minutes; the notebook reruns a few of its configurations and checks that they give the same numbers.

> The **🤔 Predict first** cells ask a question before you run something. Answer it yourself, then run the next cell.

## 1 · Setup

Only numpy and matplotlib are needed (Colab has them preinstalled). The function `num()` prints numbers in the
English format: a decimal point, and commas between thousands. That is the same point Python uses when you *type* a
decimal (`0.1`); printing only adds the thousands separators and a real minus sign.

In [ ]:
import gzip
import hashlib
import json
import math
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.ticker import FixedLocator, FuncFormatter, NullFormatter, ScalarFormatter

# The course's palette (colorblind-safe), always paired with a second encoding (marker shape, dash style, label)
BLUE, ORANGE, GREEN, GREY = "#3987e5", "#d95926", "#199e70", "#8a94a6"
DIVERGING = LinearSegmentedColormap.from_list("orange_gray_blue", [ORANGE, "#f2f2f2", BLUE])  # negative → 0 → positive
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

# Each normalization has its own color, marker shape and dash style, as in the demo
NORMS = ["none", "bn", "ln", "rms"]
LABEL = {"none": "No normalization", "bn": "BatchNorm", "ln": "LayerNorm", "rms": "RMSNorm"}
STYLE = {
    "none": {"color": GREY, "marker": "o", "mfc": "white", "ls": (0, (2, 3))},
    "bn": {"color": BLUE, "marker": "o", "mfc": BLUE, "ls": "-"},
    "ln": {"color": ORANGE, "marker": "s", "mfc": ORANGE, "ls": (0, (6, 3))},
    "rms": {"color": GREEN, "marker": "^", "mfc": GREEN, "ls": (0, (1.5, 3))},
}


def num(x, digits=2):
    """A number in the English format: 13,007 · 0.1 · −4.38."""
    x = round(float(x), digits) + 0.0  # + 0.0: never prints “−0”
    return f"{x:,.{digits}f}".replace("-", "−")


def pct(p, digits=2):
    return num(100 * p, digits) + "%"


def num_eta(e):
    """η with one significant digit: 0.3 · 0.03 · 1."""
    return num(e, max(0, -math.floor(math.log10(e) + 1e-9)))


SUP = str.maketrans("0123456789-", "⁰¹²³⁴⁵⁶⁷⁸⁹⁻")


def sci(x, digits=1):
    """A very small number: 2.7·10⁻⁶."""
    if x == 0:
        return "0"
    m, e = f"{x:.{digits}e}".split("e")
    return f"{m.replace('-', '−')}·10{str(int(e)).translate(SUP)}"


def tick(v, _pos=None):
    """An axis tick label: 0.5 · −2 · 1,000 (no trailing zeros)."""
    if abs(v) < 1e-12:
        return "0"
    s = f"{v:.4f}".rstrip("0").rstrip(".")
    return num(v, len(s.split(".")[1]) if "." in s else 0)


def fmt_axes(fig):
    """Put the number labels of every axis in the English format (text labels stay). Log axes get 1–2–5 ticks, or
    powers of 10 if the axis is very long."""
    for ax in fig.axes:
        for axis, lim in ((ax.xaxis, ax.get_xlim()), (ax.yaxis, ax.get_ylim())):
            if axis.get_scale() == "log":
                lo, hi = sorted(lim)
                mult = (1, 2, 5) if hi / lo < 300 else (1,)
                ticks = [m * 10.0**k for k in range(-4, 5) for m in mult if lo <= m * 10.0**k <= hi]
                axis.set_major_locator(FixedLocator(ticks))
                axis.set_minor_formatter(NullFormatter())
                axis.set_major_formatter(FuncFormatter(tick))
            elif isinstance(axis.get_major_formatter(), ScalarFormatter):  # a linear number axis, not text labels
                axis.set_major_formatter(FuncFormatter(tick))


print(num(13007, 0), num(0.1, 1), num(-4.3805, 4), pct(0.953213), num_eta(0.03), sci(2.7e-6), tick(1000.0))

## 2 · The data

MNIST (LeCun et al., 1998): 60,000 training images and 10,000 test images, grayscale 28×28 handwritten digits. The
cell below downloads the 4 original files and checks their MD5 hashes, so everyone who runs it gets exactly the same
data. As in the demo, we split them into three parts:
- **training:** the first 55,000 images, for computing gradients;
- **validation:** the last 5,000 images of the training file, for comparing and choosing $\eta$;
- **test:** the 10,000 images of the test file, only for reporting the final result.

The cell also loads the file of sweep results that the demo uses (`fetch_json`, with an MD5 check).

In [ ]:
URL = "https://ossci-datasets.s3.amazonaws.com/mnist/"
FILES = {  # MNIST's standard MD5 hashes
    "train-images-idx3-ubyte.gz": "f68b3c2dcbeaaa9fbdd348bbdeb94873",
    "train-labels-idx1-ubyte.gz": "d53e105ee54ea40749a09fcbcd1e9432",
    "t10k-images-idx3-ubyte.gz": "9fb629c4189551a2d022fa330f9573f3",
    "t10k-labels-idx1-ubyte.gz": "ec29112dd5afa0611ce80d1b7f02629c",
}
DATA = Path("mnist")


def fetch():
    DATA.mkdir(exist_ok=True)
    for name, md5 in FILES.items():
        p = DATA / name
        if not p.exists():
            urllib.request.urlretrieve(URL + name, p)
        assert hashlib.md5(p.read_bytes()).hexdigest() == md5, f"{name} is corrupted; delete it and download it again"


def load(kind):
    """IDX format: the first 16 bytes of an image file (8 bytes of a label file) are the header; the rest are bytes
    0–255."""
    with gzip.open(DATA / f"{kind}-images-idx3-ubyte.gz") as f:
        images = np.frombuffer(f.read(), np.uint8, offset=16).reshape(-1, 28, 28)
    with gzip.open(DATA / f"{kind}-labels-idx1-ubyte.gz") as f:
        labels = np.frombuffer(f.read(), np.uint8, offset=8)
    return images, labels


# The web demo's file: the η sweep (216 one-epoch trainings) and layer 5 of the trained BatchNorm network (the first
# card). build.py replaces COURSE_DATA_URL with the data/ folder of the public notebook repo.
def fetch_json(name, md5):
    p = Path("data") / name
    if not p.exists():
        p.parent.mkdir(exist_ok=True)
        urllib.request.urlretrieve("https://raw.githubusercontent.com/trungnguyenvn/hoc-sau-notebooks/main/data/" + name, p)
    assert hashlib.md5(p.read_bytes()).hexdigest() == md5, f"{name} is corrupted; delete it and download it again"
    return json.loads(p.read_text())


fetch()
sweep = fetch_json("mnist-norm.json", "d0fc66c752fdbd26987077feabbfc1d0")  # the demo's sweep results

In [ ]:
train_img, train_lab = load("train")
test_img, test_lab = load("t10k")
X_all, y_all = train_img.reshape(-1, 784) / 255.0, train_lab.astype(np.int64)
Xtr, ytr = X_all[:55000], y_all[:55000]  # training
Xval, yval = X_all[55000:], y_all[55000:]  # validation
Xte, yte = test_img.reshape(-1, 784) / 255.0, test_lab.astype(np.int64)  # test: for reporting only
print(f"training {num(len(ytr), 0)} · validation {num(len(yval), 0)} · test {num(len(yte), 0)} images")
n_runs = sum(len(r["seeds"]) for r in sweep["runs"])
print(
    f"sweep file: {len(sweep['norms'])} normalizations × {len(sweep['depths'])} depths × {len(sweep['etas'])} values "
    f"of η × {len(sweep['seeds'])} seeds = {n_runs} training runs of 1 epoch"
)
assert (len(ytr), len(yval), len(yte)) == (sweep["nTrain"], sweep["nVal"], sweep["nTest"])

The first eight test images. In section 7 we will look at the network's real activations on exactly these eight
images, as in the demo:

In [ ]:
fig, axes = plt.subplots(1, 8, figsize=(8, 1.4))
for i, ax in enumerate(axes):
    ax.imshow(test_img[i], cmap="gray_r")
    ax.set_title(f"image {i + 1} ({test_lab[i]})", fontsize=9)
    ax.axis("off")
plt.show()

## 3 · Lesson 09's deep network, and the input to a hidden layer

We reuse exactly Lesson 09's network: 784 → $L$ hidden layers × 64 neurons → 10, ReLU, He initialization
($W \sim \mathcal N(0, 2/n_{in})$), biases at 0. As in the demo, the random numbers are generated with `mulberry32`
(the JavaScript generator the demo uses) and then turned into normally distributed numbers by the Box–Muller
transform, so seed 1 gives exactly the network that Lesson 09's demo draws, and a depth $L$ is just “take the first
$L$ layers”. The cell below builds the 10-layer network and counts its parameters.

In [ ]:
WIDTH, BATCH, N_TRAIN = 64, 64, 55000
EPS, MOM = 1e-5, 0.1  # the normalization's ε; the coefficient of the running mean (section 8)


def mulberry32(seed, n):
    """The first n numbers of JavaScript's mulberry32(seed) generator, divided by 2³² to make floats in [0, 1)."""
    a = ((seed + np.arange(1, n + 1, dtype=np.uint64) * 0x6D2B79F5) & 0xFFFFFFFF).astype(np.uint32)
    t = (a ^ (a >> 15)) * (a | 1)
    t = (t + (t ^ (t >> 7)) * (t | 61)) ^ t
    return (t ^ (t >> 14)).astype(np.float64) / 2**32


def init(norm, depth, seed=1):
    """The network 784 → depth×64 → 10 with the normalization `norm` ∈ {none, bn, ln, rms}; γ = 1, β = 0 at first."""
    shapes = [(WIDTH, 10), (784, WIDTH)] + [(WIDTH, WIDTH)] * (depth - 1)  # drawn in the same order as the demo
    sizes = [r * c for r, c in shapes]
    u = mulberry32(seed, 2 * sum(sizes))
    z = np.sqrt(-2 * np.log(1 - u[0::2])) * np.cos(2 * np.pi * u[1::2])  # Box–Muller: each uniform pair → one normal
    e = [part.reshape(s) for part, s in zip(np.split(z, np.cumsum(sizes)[:-1]), shapes)]
    W = [np.sqrt(2 / r) * ei for (r, _), ei in zip(shapes[1:], e[1:])] + [np.sqrt(2 / WIDTH) * e[0]]  # He
    return {
        "norm": norm,
        "depth": depth,
        "W": W,
        "b": [np.zeros(w.shape[1]) for w in W],
        "g": [np.ones(WIDTH) for _ in range(depth)],  # γ of each hidden layer
        "be": [np.zeros(WIDTH) for _ in range(depth)],  # β (RMSNorm doesn't use it)
        "rm": [np.zeros(WIDTH) for _ in range(depth)],  # BatchNorm's running mean (section 8)
        "rv": [np.ones(WIDTH) for _ in range(depth)],  # BatchNorm's running variance
    }


P10 = init("none", 10)
n_par = sum(w.size + b.size for w, b in zip(P10["W"], P10["b"]))
print(f"network with 10 hidden layers: {len(P10['W'])} W matrices, {num(n_par, 0)} parameters (not counting γ, β)")
print("W₁:", P10["W"][0].shape, " W₂…W₁₀:", P10["W"][1].shape, " output W:", P10["W"][-1].shape)
assert n_par == 784 * 64 + 64 + 9 * (64 * 64 + 64) + 64 * 10 + 10

Each hidden layer computes $a = hW + b$, then $h \leftarrow \mathrm{ReLU}(a)$. We call $a$ the **input to the
normalization layer**: this lesson inserts a normalization step in between, giving Linear → normalization → ReLU. The
cell below runs the first 8 test images through the first 4 layers of the untrained network, then prints $a$ of the
5th layer for 6 of the 64 neurons. Each row is an image, each column a neuron:

In [ ]:
h = Xte[:8]
for W in P10["W"][:4]:
    h = np.maximum(0, h @ W)  # the first 4 hidden layers (b = 0 at initialization)
a5 = h @ P10["W"][4]  # the input to the 5th normalization layer: 8 images × 64 neurons
print("a of layer 5 (8 images × the first 6 neurons):")
for i, row in enumerate(a5[:, :6]):
    print(f"  image {i + 1} (digit {yte[i]}): " + "  ".join(f"{num(v, 2):>6}" for v in row))
mu_col, sd_col = a5.mean(0), a5.std(0)
print(f"column means (64 neurons) from {num(mu_col.min())} to {num(mu_col.max())}; ", end="")
print(f"standard deviations from {num(sd_col.min())} to {num(sd_col.max())}")

Each neuron has its own scale: the first column is negative for all 8 images, the second column is positive for all
8 images, and the standard deviations of the columns differ several-fold. The next layer receives exactly these
numbers. During training the earlier layers change, so the scale of each column changes with them (section 6
measures this).

## 4 · Subtract the mean, divide by the standard deviation: three choices of axis

Normalization brings a group of numbers to mean 0 and variance 1, then lets the network learn back a scale $\gamma$
and a shift $\beta$ of its own:

$$\hat x = \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}} \qquad y = \gamma\,\hat x + \beta$$

($\sigma^2$ here is the variance, not the sigmoid function; $\epsilon = 10^{-5}$ avoids dividing by 0.) The three
methods differ only in **which group of numbers of the matrix $a$ they take $\mu, \sigma^2$ over** (rows = images,
columns = neurons):

- **BatchNorm** (Ioffe & Szegedy, 2015): by **column**, that is, each neuron over the images of the batch.
- **LayerNorm** (Ba et al., 2016): by **row**, that is, each image over its 64 neurons, without needing any other
  image.
- **RMSNorm** (Zhang & Sennrich, 2019): also by row, but it **doesn't subtract the mean** and has no $\beta$; it only
  divides by the root mean square, $\mathrm{RMS}(x) = \sqrt{\tfrac1d \sum_i x_i^2 + \epsilon}$.

We do the first column by hand with a loop, then write the vectorized version:

In [ ]:
col = a5[:, 0]
mu = sum(col) / len(col)
var = sum((v - mu) ** 2 for v in col) / len(col)  # the variance “divided by n”, as in the paper
xhat_loop = [(v - mu) / math.sqrt(var + EPS) for v in col]
print(f"column 1: μ = {num(mu, 4)}, σ² = {num(var, 4)} → x̂ =", " ".join(num(v, 2) for v in xhat_loop))
print(f"after normalization: mean {num(np.mean(xhat_loop), 4)}, standard deviation {num(np.std(xhat_loop), 4)}")


def norm_forward(a, kind, gamma, beta):
    """BatchNorm (kind='bn', by column), LayerNorm ('ln', by row) or RMSNorm ('rms'). Returns y and the cache for the
    backward pass."""
    if kind == "rms":
        r = np.sqrt((a * a).mean(1, keepdims=True) + EPS)  # the RMS of each row
        xhat = a / r
        return gamma * xhat, (xhat, r)  # RMSNorm only has γ
    axis = 0 if kind == "bn" else 1  # bn: the mean over the images; ln: over the neurons of one image
    inv = 1 / np.sqrt(a.var(axis, keepdims=True) + EPS)
    xhat = (a - a.mean(axis, keepdims=True)) * inv
    return gamma * xhat + beta, (xhat, inv)


one, zero = np.ones(WIDTH), np.zeros(WIDTH)
y_bn, y_ln, y_rms = (norm_forward(a5, k, one, zero)[0] for k in ("bn", "ln", "rms"))
assert np.allclose(y_bn[:, 0], xhat_loop)  # vectorized version = loop
for name, y in [("BatchNorm", y_bn), ("LayerNorm", y_ln), ("RMSNorm", y_rms)]:
    c_mu, r_mu, r_sd = np.abs(y.mean(0)).max(), np.abs(y.mean(1)).max(), y.std(1)
    r_rms = np.sqrt((y * y).mean(1))
    print(
        f"{name:>9}: |column mean| ≤ {num(c_mu, 3)} · |row mean| ≤ {num(r_mu, 3)} · "
        f"row standard deviation {num(r_sd.min(), 3)}–{num(r_sd.max(), 3)} · "
        f"row RMS {num(r_rms.min(), 3)}–{num(r_rms.max(), 3)}"
    )

The loop and the vectorized version give the same column. After normalization, that column has mean 0 and standard
deviation 0.9996, not exactly 1: its variance is only 0.0118, so $\epsilon = 10^{-5}$ starts to have an effect. The
last three lines read as follows:

- **BatchNorm** brings every **column** to mean 0, but the individual rows (images) are still off center.
- **LayerNorm** brings every **row** to mean 0 and standard deviation 1; the columns, not.
- **RMSNorm** only makes each row have RMS = 1; the row's mean is still nonzero. By dropping the mean subtraction, it
  is one computation per row cheaper than LayerNorm.

## 5 · The backward pass, and a check with finite differences

To train, we need $\partial L/\partial a$ given $\partial L/\partial y$. The hard part is that $a$ affects $\hat x$
along three paths: directly, through $\mu$ and through $\sigma$. Let $g = \gamma \odot \partial L/\partial y$ (the
gradient with respect to $\hat x$), and let a bar on top denote the mean over the normalized group (the column for
BatchNorm, the row for LayerNorm). Differentiating along all three paths and simplifying gives:

$$\frac{\partial L}{\partial a} = \frac{1}{\sqrt{\sigma^2+\epsilon}}\Big(g - \overline{g} - \hat x \odot \overline{g \odot \hat x}\Big)
\qquad \frac{\partial L}{\partial \gamma} = \sum_{\text{images}} \frac{\partial L}{\partial y} \odot \hat x
\qquad \frac{\partial L}{\partial \beta} = \sum_{\text{images}} \frac{\partial L}{\partial y}$$

The term $\overline g$ comes from $\mu$, the last term from $\sigma$. RMSNorm doesn't subtract $\mu$, so it only
loses $\overline g$ and replaces $\sigma$ with the RMS. This formula is exact even with $\epsilon$. The cell below
writes it as code, then compares it with the numerical gradient $\big(L(a+h) - L(a-h)\big)/2h$ on a small random
matrix, with $L = \sum R \odot y$ for an arbitrary matrix $R$.

In [ ]:
def norm_backward(dy, kind, cache, gamma):
    """The backward pass of norm_forward: from dL/dy to dL/da, dL/dγ, dL/dβ."""
    xhat, s = cache  # s = 1/σ (BatchNorm, LayerNorm) or the RMS (RMSNorm)
    dgamma, dbeta = (dy * xhat).sum(0), dy.sum(0)  # γ, β belong to each neuron: sum over the images
    g = dy * gamma  # dL/dx̂
    if kind == "rms":  # no μ: only the direct path and the path through the RMS remain
        return (g - xhat * (g * xhat).mean(1, keepdims=True)) / s, dgamma, dbeta
    axis = 0 if kind == "bn" else 1
    m = g.shape[axis]  # the number of values in each group: m·ḡ = the sum of g
    da = s / m * (m * g - g.sum(axis, keepdims=True) - xhat * (g * xhat).sum(axis, keepdims=True))
    return da, dgamma, dbeta


def numgrad(f, x, h=1e-6):
    """The numerical gradient of f() with respect to the array x (changes x in place, then puts it back)."""
    out = np.zeros_like(x)
    for i in np.ndindex(x.shape):
        old = x[i]
        x[i] = old + h
        up = f()
        x[i] = old - h
        down = f()
        x[i] = old
        out[i] = (up - down) / (2 * h)
    return out


def rel_err(u, v):
    return float(np.abs(u - v).max() / (np.abs(u).max() + np.abs(v).max()))


rng = np.random.default_rng(0)
a_t, R = rng.normal(1.0, 2.0, (6, 5)), rng.normal(size=(6, 5))  # 6 “images” × 5 “neurons”
g_t, b_t = rng.normal(1.0, 0.3, 5), rng.normal(0.0, 0.3, 5)
worst = 0.0
for kind in ("bn", "ln", "rms"):
    da, dg, dbe = norm_backward(R, kind, norm_forward(a_t, kind, g_t, b_t)[1], g_t)

    def L(kind=kind):
        return float((R * norm_forward(a_t, kind, g_t, b_t)[0]).sum())

    errs = [rel_err(da, numgrad(L, a_t)), rel_err(dg, numgrad(L, g_t))]
    errs += [] if kind == "rms" else [rel_err(dbe, numgrad(L, b_t))]  # RMSNorm has no β
    worst = max(worst, *errs)
    print(f"{LABEL[kind]:>9}: relative error of dL/da, dL/dγ" + ("" if kind == "rms" else ", dL/dβ"), end=": ")
    print(" · ".join(sci(e) for e in errs))
assert worst < 1e-7, "the backward pass is wrong: check the formula again"

Relative errors of order $10^{-10}$ (the limit of finite differences), for all three methods and all three
gradients: the formula is right.

Note the line `dbeta = dy.sum(0)` and the term $\overline g$: with BatchNorm, $\partial L/\partial a$ summed over a
column is always 0. So the bias $b$ of the Linear layer right before BatchNorm **never receives a gradient**:
BatchNorm subtracts the mean, so it cancels any $b$, and $\beta$ does its job instead (Ioffe & Szegedy, 2015,
section 3.2). That is why $b$ is usually dropped before BatchNorm (`bias=False` in PyTorch). We will see this on a
real network in section 7.

## 6 · Inserting them into the network, and measuring the “drift”

Each hidden layer is now $a = hW + b$ → normalization → ReLU, exactly as in `reel/mnist_mlp/norm.py`, the script
that makes the demo's data:
- SGD, batch 64, 1 pass (epoch) through the 55,000 training images, that is, 860 steps;
- the seed $s$ decides both the initial weights (`mulberry32(s)`) and the order of the images (`default_rng(s)`);
- BatchNorm at training time uses the batch's statistics, and also updates a running mean that section 8 will
  explain;
- a run “blows up” when the loss of a batch is no longer a finite number.

The function `forward` has an extra option `shake` for the experiment at the end of this section; ignore it for now.

In [ ]:
def normalize(P, l, a, mode):
    """The l-th normalization layer. mode = 'train': BatchNorm uses the batch's statistics and updates the running mean;
    'eval': uses the running mean; 'batch': the batch's statistics, without updating. LayerNorm and RMSNorm do the
    same in every mode."""
    kind = P["norm"]
    if kind == "bn" and mode == "eval":
        xhat = (a - P["rm"][l]) * (1 / np.sqrt(P["rv"][l] + EPS))
        return P["g"][l] * xhat + P["be"][l], None
    if kind == "bn" and mode == "train":
        m = len(a)
        P["rm"][l] = (1 - MOM) * P["rm"][l] + MOM * a.mean(0)
        P["rv"][l] = (1 - MOM) * P["rv"][l] + MOM * a.var(0) * m / (m - 1)  # the unbiased variance, as in PyTorch
    return norm_forward(a, kind, P["g"][l], P["be"][l])


def forward(P, x, mode="train", shake=None):
    h, cache = x, []
    for l in range(P["depth"]):
        a = h @ P["W"][l] + P["b"][l]
        y, c = (a, None) if P["norm"] == "none" else normalize(P, l, a, mode)
        s = 1.0
        if shake:  # (noise level, rng): multiply each neuron by, and add to it, a new random number at every step
            level, rng = shake
            s = np.exp(level * rng.normal(size=WIDTH))
            y = y * s + level * rng.normal(size=WIDTH)
        cache.append((h, c, y, s))
        h = np.maximum(0, y)
    return cache, h, h @ P["W"][-1] + P["b"][-1]


def softmax(z):
    e = np.exp(z - z.max(1, keepdims=True))
    return e / e.sum(1, keepdims=True)


def cross_entropy(z, y):
    zmax = z.max(1)
    return float((zmax + np.log(np.exp(z - zmax[:, None]).sum(1)) - z[np.arange(len(y)), y]).mean())


def backward(P, cache, h, z, y):
    """The gradients with respect to W, b, γ, β of every layer (Lesson 06, plus norm_backward)."""
    dz = softmax(z)
    dz[np.arange(len(y)), y] -= 1
    dz /= len(y)
    G = {"W": [h.T @ dz], "b": [dz.sum(0)], "g": [], "be": []}
    dh = dz @ P["W"][-1].T
    for l in range(P["depth"] - 1, -1, -1):
        h_in, c, y_l, s = cache[l]
        dy = dh * (y_l > 0) * s  # through the ReLU (and through the multiplicative noise, if any)
        if P["norm"] == "none":
            da = dy
        else:
            da, dg, dbe = norm_backward(dy, P["norm"], c, P["g"][l])
            G["g"].insert(0, dg)
            G["be"].insert(0, dbe)
        G["W"].insert(0, h_in.T @ da)
        G["b"].insert(0, da.sum(0))
        dh = da @ P["W"][l].T
    return G


def train(norm, depth, eta, seed=1, steps=860, shake=0.0, watch=None):
    """SGD with batch 64 on the training images, in the seed's order. Returns (network, blow-up step or None).
    watch(step, P, G), if given, is called at step 0 and after each update step."""
    P = init(norm, depth, seed)
    order = np.random.default_rng(seed).permutation(N_TRAIN)
    rng = np.random.default_rng(1000 + seed) if shake else None
    if watch:
        watch(0, P, None)
    with np.errstate(all="ignore"):  # a run that blows up overflows; we detect that ourselves below
        for step in range(1, steps + 1):
            idx = order[(step - 1) * BATCH : step * BATCH]
            cache, h, z = forward(P, Xtr[idx], "train", (shake, rng) if shake else None)
            if not np.isfinite(cross_entropy(z, ytr[idx])):
                return P, step
            G = backward(P, cache, h, z, ytr[idx])
            for k in ("W", "b", "g", "be"):
                if k == "be" and norm == "rms":
                    continue  # RMSNorm has no β
                for p, g in zip(P[k], G[k]):
                    p -= eta * g
            if watch:
                watch(step, P, G)
    return P, None


def accuracy(P, X, y, mode="eval"):
    hits = [forward(P, X[i : i + 2000], mode)[2].argmax(1) == y[i : i + 2000] for i in range(0, len(X), 2000)]
    return float(np.concatenate(hits).mean())


def dead_share(P):
    """The fraction of hidden neurons that output 0 for all of the first 500 validation images (a “completely dead”
    ReLU), as in the demo."""
    cache = forward(P, Xval[:500], "eval")[0]
    return float(np.mean([(y <= 0).all(0).mean() for _, _, y, _ in cache]))


print(
    f"1 epoch = {math.ceil(N_TRAIN / BATCH)} steps; the last step has only {N_TRAIN % BATCH} images, as in the demo's script"
)

Now we measure what the BatchNorm paper calls **internal covariate shift**: the distribution of a layer's inputs
changes as the layers before it are updated. We train two 10-layer networks, seed 1, each with an $\eta$ at which it
learns well (according to the demo's sweep table):
- no normalization, $\eta = 0.03$;
- BatchNorm, $\eta = 0.3$. This is also the network the demo uses for the card “Normalize along which axis?”, so
  section 7 reuses it.

Every 43 steps, we pass 500 validation images through the network and record the mean and the standard deviation of
$a$ at layer 5 for the first 4 neurons. For the BatchNorm network, we also record the output $y$ of the normalization
layer, that is, what the next layer actually receives (normalized with the statistics of those same 500 images).

In [ ]:
def drift_probe(log):
    def watch(step, P, G):
        if step % 43:
            return
        cache = forward(P, Xval[:500], "batch")[0]  # "batch": leaves the running mean alone
        h_in, _, y, _ = cache[4]
        a = h_in @ P["W"][4] + P["b"][4]
        log.append((step, a.mean(0)[:4], a.std(0)[:4], y.mean(0)[:4], y.std(0)[:4]))

    return watch


drift = {"none": [], "bn": []}
P_none, _ = train("none", 10, 0.03, watch=drift_probe(drift["none"]))
P_bn, _ = train("bn", 10, 0.3, watch=drift_probe(drift["bn"]))
for kind in drift:
    (s0, m0, sd0, ym0, ysd0), (s1, m1, sd1, ym1, ysd1) = drift[kind][0], drift[kind][-1]
    print(f"{LABEL[kind]}: a of 4 neurons, step 0 → step {s1}")
    print("   mean:", " · ".join(f"{num(u)} → {num(v)}" for u, v in zip(m0, m1)))
    print("   standard deviation:", " · ".join(f"{num(u)} → {num(v)}" for u, v in zip(sd0, sd1)))
    if kind == "bn":
        print("   output y after BatchNorm, mean:", " · ".join(f"{num(u)} → {num(v)}" for u, v in zip(ym0, ym1)))
        print(
            "   output y after BatchNorm, standard deviation:",
            " · ".join(f"{num(u)} → {num(v)}" for u, v in zip(ysd0, ysd1)),
        )
val_none, val_bn = accuracy(P_none, Xval, yval), accuracy(P_bn, Xval, yval)
print(f"after 1 epoch, validation accuracy: no normalization {pct(val_none)} · BatchNorm {pct(val_bn)}")

The figure below plots those numbers step by step. The top row is the mean, the bottom row the standard deviation
over the 500 validation images. Each neuron has its own color and marker shape. The three columns are $a$ of the
network without normalization, $a$ of the BatchNorm network (before normalization), and $y$ of the BatchNorm network
(after normalization, what the next layer receives).

In [ ]:
NEURON = [(BLUE, "o"), (ORANGE, "s"), (GREEN, "^"), (GREY, "D")]
panels = [
    ("No normalization: a", drift["none"], 1, 2),
    ("BatchNorm: a (before normalization)", drift["bn"], 1, 2),
    ("BatchNorm: y (after normalization)", drift["bn"], 3, 4),
]
fig, axes = plt.subplots(2, 3, figsize=(10, 4.6), sharex=True, sharey="row")
for j, (title, log, im, isd) in enumerate(panels):
    steps = [r[0] for r in log]
    for k, (color, mk) in enumerate(NEURON):
        for row, idx in ((0, im), (1, isd)):
            axes[row, j].plot(
                steps, [r[idx][k] for r in log], color=color, marker=mk, ms=3.5, lw=1.3, label=f"neuron {k + 1}"
            )
    axes[0, j].set_title(title, fontsize=10)
    axes[1, j].set_xlabel("SGD step")
    for row in (0, 1):
        axes[row, j].grid(alpha=0.3)
axes[0, 0].set_ylabel("mean")
axes[1, 0].set_ylabel("standard deviation")
axes[1, 0].legend(frameon=False, fontsize=8, ncol=2)
fmt_axes(fig)
plt.tight_layout()
plt.show()

In the network without normalization, the standard deviation of all four neurons grows several-fold within one
epoch, and the mean shifts too. Layer 6 has to learn on an input whose scale keeps changing. In the BatchNorm
network, $a$ still drifts (middle column), but $y$ is held at a mean near 0 and a standard deviation near 1 (right
column): BatchNorm blocks the first two moments of the drift.

Ioffe & Szegedy (2015) argue that this is exactly why BatchNorm helps training. Santurkar et al. (2018) tested this
hypothesis with a reverse experiment: deliberately **adding** drift right after BatchNorm. We repeat that experiment.
At each step, each neuron is multiplied by a number $e^{0.3\,z_1}$ and has $0.3\,z_2$ added to it, with
$z_1, z_2 \sim \mathcal N(0, 1)$ drawn anew at every step. At evaluation time there is no noise.

In [ ]:
z = np.random.default_rng(0).normal(size=(2, 100_000))
lo, hi = np.quantile(np.exp(0.3 * z[0]), [0.025, 0.975])
print(
    f"noise level 0.3: 95% of the time, the multiplier lies in [{num(lo)}, {num(hi)}] and the added number within "
    f"±{num(0.3 * 1.96)}"
)
y_shift = max(abs(v) for r in drift["bn"] for v in r[3])
y_sd = [v for r in drift["bn"] for v in r[4]]
print(f"natural drift of y after BatchNorm over the whole epoch: |mean| ≤ {num(y_shift)}, ", end="")
print(f"standard deviation {num(min(y_sd))}–{num(max(y_sd))}")

### 🤔 Predict first
This noise creates a drift many times larger than the natural drift just measured, and it changes at every step. If
BatchNorm helps because it blocks drift, the BatchNorm network with noise should learn much worse. From the level of
clean BatchNorm, what do you guess the validation accuracy will drop to: close to the network without
normalization, close to random guessing, or will it barely change?

In [ ]:
P_shake, boom = train("bn", 10, 0.3, shake=0.3)
val_shake = accuracy(P_shake, Xval, yval)
print(f"BatchNorm, η = 0.3, noise of level 0.3 after each normalization layer: validation accuracy {pct(val_shake)}")
print(f"clean BatchNorm: {pct(val_bn)} · no normalization (η = 0.03): {pct(val_none)}")

It barely changes: adding strong noise right after every normalization layer makes the BatchNorm network only
slightly worse, and it is still better than the network without normalization we just trained (one seed; section 12
tries two more seeds). This network learns well even though the distribution of every layer's inputs gets scrambled
at **every step**, so blocking drift cannot be the whole reason BatchNorm helps. Santurkar et al. (2018) found
exactly this on a large image network (VGG on CIFAR-10). They proposed a different explanation: BatchNorm makes the
**loss landscape smoother**, so the gradients are more stable and tolerate larger steps. To this day, why BatchNorm
works is still debated; section 9 measures one specific mechanism.

## 7 · Normalize along which axis? On real activations

The BatchNorm network we just trained (10 layers, $\eta = 0.3$, seed 1) is exactly the network of the demo's first
card. The cell below takes the input $a$ to the 5th normalization layer for the first 8 test images, then compares
it with the numbers the demo stores (8 images × the first 8 neurons).

In [ ]:
pan = sweep["panel"]
h_in = forward(P_bn, Xte[:8], "eval")[0][4][0]  # the input to hidden layer 5 (after layer 4's ReLU)
A = h_in @ P_bn["W"][4] + P_bn["b"][4]  # 8 images × 64 neurons, before normalization
test_bn = accuracy(P_bn, Xte, yte)
print(
    f"BatchNorm network: validation accuracy {pct(val_bn)}, test {pct(test_bn)} "
    f"(demo: {pct(pan['val'])}, {pct(pan['testRunning'])})"
)
print(
    "a of layer 5, 8 images × the first 8 neurons; largest difference from the demo:",
    num(np.abs(A[:, :8] - pan["a"]).max(), 5),
)
assert abs(val_bn - pan["val"]) < 5e-4 and abs(test_bn - pan["testRunning"]) < 5e-4, "differs from the demo"
assert np.allclose(A[:, :8], pan["a"], atol=1e-3), "differs from the demo: check the data again"
for key, arr in [
    ("runMean", P_bn["rm"][4]),
    ("runVar", P_bn["rv"][4]),
    ("gamma", P_bn["g"][4]),
    ("beta", P_bn["be"][4]),
]:
    assert np.allclose(arr[:8], pan[key], atol=1e-3), f"{key} differs from the demo"
print("the running mean, running variance, γ and β of layer 5 match the demo too")

The figure below colors the same 8 × 8 cells before and after the three normalizations: blue is positive, orange is
negative, the darker the farther from 0, and each cell shows its value. The black frame encloses the cells that share
one pair $\mu, \sigma$ with the top-left cell:
- BatchNorm normalizes by **column**, over this batch of 8 images;
- LayerNorm and RMSNorm normalize by **row**, over all 64 neurons of each image (the figure only draws the first 8
  neurons; the demo computes over the 8 neurons it shows).

In [ ]:
views = [
    ("a (not normalized)", A, None),
    ("BatchNorm, batch of 8 images: by column", norm_forward(A, "bn", one, zero)[0], "col"),
    ("LayerNorm: by row", norm_forward(A, "ln", one, zero)[0], "row"),
    ("RMSNorm: by row, μ not subtracted", norm_forward(A, "rms", one, zero)[0], "row"),
]
vmax = max(np.abs(v[:, :8]).max() for _, v, _ in views)
fig, axes = plt.subplots(2, 2, figsize=(9.6, 8.4))
for ax, (title, v, group) in zip(axes.flat, views):
    im = ax.imshow(v[:, :8], cmap=DIVERGING, vmin=-vmax, vmax=vmax)
    for i in range(8):
        for j in range(8):
            ax.text(j, i, num(v[i, j], 2), ha="center", va="center", fontsize=7)
    if group:  # frame: the cells that share μ, σ with cell (1, 1)
        w, hgt = (1, 8) if group == "col" else (8, 1)
        ax.add_patch(plt.Rectangle((-0.5, -0.5), w, hgt, fill=False, ec="black", lw=2.2))
    ax.set_xticks(range(8), [f"n{j + 1}" for j in range(8)], fontsize=8)
    ax.set_yticks(range(8), [f"image {i + 1} ({yte[i]})" for i in range(8)], fontsize=8)
    ax.set_title(title, fontsize=10)
fig.colorbar(im, ax=axes, fraction=0.03, pad=0.02, label="value")
fmt_axes(fig)
plt.show()
for title, v, _ in views:
    neg = [f"n{j + 1}" for j in range(8) if (v[:, j] < 0).all()]
    pos = [f"n{j + 1}" for j in range(8) if (v[:, j] > 0).all()]
    print(
        f"{title}: columns negative for all 8 images: {', '.join(neg) or 'none'}; "
        f"columns positive for all 8 images: {', '.join(pos) or 'none'}"
    )

In the top-left panel, each column has its own scale: column n5 is negative (orange) for all 8 images, while column
n3 has a few cells many times larger than the others. After BatchNorm, within this batch of 8 images, no column keeps
the same sign for all 8 images: each neuron is set to mean 0 *relative to the other images in the batch*. After
LayerNorm or RMSNorm, each row is put on the same scale, but column n5 is still negative for all 8 images. (The eight
cells of a row don't have mean 0 because the computation uses all 64 neurons, while the figure only draws 8.)

The consequence of the axis: the BatchNorm result for an image **depends on the other images in the batch**, while
LayerNorm and RMSNorm don't. That is why LayerNorm suits the Transformer, where each token is normalized on its own
and the batch can be very small.

### γ, β: giving the network its freedom back

Forcing every neuron to mean 0 and variance 1 can take away something the network needs. That is why each neuron
also gets a learned $\gamma$ and $\beta$. Ioffe & Szegedy (2015) point out that with BatchNorm, setting
$\gamma = \sqrt{\sigma^2 + \epsilon}$ and $\beta = \mu$ gives back exactly the original values. The cell below checks
this with $\mu, \sigma^2$ taken as the running mean and variance (section 8), looks at how the network has learned
$\gamma, \beta$, and tries the same thing with LayerNorm.

In [ ]:
xhat_bn = (A - P_bn["rm"][4]) / np.sqrt(P_bn["rv"][4] + EPS)
undo = np.sqrt(P_bn["rv"][4] + EPS) * xhat_bn + P_bn["rm"][4]
print(f"BatchNorm with γ = √(σ² + ε), β = μ: largest difference from a = {sci(np.abs(undo - A).max())}")
assert np.abs(undo - A).max() < 1e-9
gs, bs = np.concatenate(P_bn["g"]), np.concatenate(P_bn["be"])
print(f"learned γ (10 layers × 64 neurons): {num(gs.min())} to {num(gs.max())}; β: {num(bs.min())} to {num(bs.max())}")

# LayerNorm: γ, β belong to each neuron, while μ, σ belong to each image. Find the best γ, β (least squares) on the
# 10,000 test images to bring LayerNorm's x̂ back to a:
A_all = forward(P_bn, Xte, "eval")[0][4][0] @ P_bn["W"][4] + P_bn["b"][4]
xl = norm_forward(A_all, "ln", one, zero)[0]
g_fit = ((xl - xl.mean(0)) * (A_all - A_all.mean(0))).mean(0) / xl.var(0)
b_fit = A_all.mean(0) - g_fit * xl.mean(0)
resid = np.linalg.norm(A_all - (g_fit * xl + b_fit)) / np.linalg.norm(A_all - A_all.mean(0))
print(f"LayerNorm with the best γ, β for each neuron: still off by {pct(resid, 0)} (relative to the spread of a)")

With BatchNorm the “undo” is exact, so normalization takes nothing away from the network: if it needs to, it can
learn the original values back. The real network doesn't do that: after 1 epoch, $\gamma$ is still around 1 and
$\beta$ around 0, that is, it *keeps* the normalization. With LayerNorm, undoing is impossible: each image loses its
own mean and standard deviation, while $\gamma, \beta$ have only one value per neuron, shared by every image.

One more detail from section 5: the bias $b$ of the Linear layer right before BatchNorm never receives a gradient.
A check on the network we just trained for 860 steps:

In [ ]:
b_max = max(np.abs(b).max() for b in P_bn["b"][:-1])  # the 10 hidden layers; the output layer has no BatchNorm
print(
    f"largest |b| in the 10 hidden layers after 860 steps: {sci(b_max)} (only rounding error); "
    f"output layer: {num(np.abs(P_bn['b'][-1]).max(), 3)}"
)
assert b_max < 1e-12

## 8 · BatchNorm at training time and at inference time

At training time, BatchNorm uses the $\mu, \sigma^2$ of the batch of 64 images. At inference time, we want the
prediction for an image to depend only on that image. That is why, during training, each layer also keeps a
**running mean**:

$$\mu_{\text{running}} \leftarrow 0.9\,\mu_{\text{running}} + 0.1\,\mu_{\text{batch}} \qquad
\sigma^2_{\text{running}} \leftarrow 0.9\,\sigma^2_{\text{running}} + 0.1\,\tfrac{m}{m-1}\sigma^2_{\text{batch}}$$

with $m = 64$ images per batch; the factor $\tfrac{m}{m-1}$ gives the unbiased variance, as in Ioffe & Szegedy
(2015) and PyTorch. PyTorch calls the coefficient 0.1 `momentum`, although it is the weight of the *new* value, the
opposite of Lesson 07's momentum. At inference time (`model.eval()` in PyTorch), BatchNorm uses those two running
values: the `"eval"` mode of the function `normalize`.

The cell below writes out the *wrong* way of doing inference: still using the batch's statistics, with batches of
$B$ consecutive test images.

In [ ]:
def predict_batch_stats(P, X, B):
    """The logits of the BatchNorm network with the wrong kind of inference: each group of B consecutive images is
    normalized with its own statistics."""
    h, n = X, len(X) // B * B
    for l in range(P["depth"]):
        a = h @ P["W"][l] + P["b"][l]
        groups = [a[:n].reshape(-1, B, WIDTH)] + ([a[n:][None]] if n < len(a) else [])  # the last group may be smaller
        xhat = [(g - g.mean(1, keepdims=True)) / np.sqrt(g.var(1, keepdims=True) + EPS) for g in groups]
        h = np.maximum(0, P["g"][l] * np.concatenate([x.reshape(-1, WIDTH) for x in xhat]) + P["be"][l])
    return h @ P["W"][-1] + P["b"][-1]


z64 = predict_batch_stats(P_bn, Xte, 64)
print(
    f"batch of 64 images, the batch's statistics: {pct((z64.argmax(1) == yte).mean())} correct · "
    f"running mean: {pct(test_bn)} correct"
)

### 🤔 Predict first
Now we forget about inference mode entirely and pass the images through the network **one at a time** (a batch of
1 image), still using the batch's statistics. On the 10,000 test images, what percentage does the network get
right? (Hint: with one image, what is the $\mu$ of each neuron?)

In [ ]:
z1 = predict_batch_stats(P_bn, Xte, 1)
same = np.allclose(z1, z1[0])
print(f"batch of 1 image: every image gives the same logits: {same}; the network always guesses {z1[0].argmax()}")
acc_b = {B: float((predict_batch_stats(P_bn, Xte, B).argmax(1) == yte).mean()) for B in (1, 2, 8, 64)}
for B, acc in acc_b.items():
    print(f"  batch statistics, B = {B:>2}: {pct(acc)} correct (demo: {pct(pan['testBatch'][str(B)])})")
print(f"  running mean (the right way): {pct(test_bn)}")
print(f"test images that are a {z1[0].argmax()}: {num((yte == z1[0].argmax()).sum(), 0)} / {num(len(yte), 0)}")
for B, acc in acc_b.items():
    assert abs(acc - pan["testBatch"][str(B)]) < 5e-4, "differs from the demo: check the data again"

With one image, $\mu$ is that very value, so $x - \mu = 0$ at every neuron: the image vanishes right at the first
layer. Every image gives the same output (only $\beta$ gets through), so the network always guesses the same digit.
Its accuracy is exactly the share of images of that digit in the test set. A batch of 2 images is bad too: each
normalized value is only $\pm 1$, that is, “larger or smaller than the other image”. The smaller the batch, the
noisier the statistics; even a batch of 64 images is still worse than the running mean. This is a real, common bug:
forgetting `model.eval()` in PyTorch makes a BatchNorm network do inference with the batch's statistics.

The running mean is only an estimate too: it accumulates the $\mu$ of the batches that went through, while the
weights were still changing. Wu & Johnson (2021) propose “PreciseBN”: freeze the weights, then compute the true
$\mu, \sigma^2$ over many training images. The cell below does that on all 55,000 images, one layer at a time:

In [ ]:
P_precise = {**P_bn, "rm": [], "rv": []}
h = Xtr
for l in range(10):
    a = h @ P_bn["W"][l] + P_bn["b"][l]
    P_precise["rm"].append(a.mean(0))
    P_precise["rv"].append(a.var(0))
    h = np.maximum(0, P_bn["g"][l] * (a - a.mean(0)) / np.sqrt(a.var(0) + EPS) + P_bn["be"][l])
test_precise = accuracy(P_precise, Xte, yte)
d_mu = np.abs(P_precise["rm"][4] - P_bn["rm"][4]).max()
print(f"layer 5: the running mean is off by at most {num(d_mu, 3)} from the true mean over the training set")
print(f"test accuracy: running mean {pct(test_bn)} · true statistics over 55,000 images {pct(test_precise)}", end=" ")
print(f"(a difference of {num(100 * (test_precise - test_bn), 2)} percentage points)")

The true statistics give almost the same result as the running mean (a difference of 0.16 percentage points, on one
network and one seed): here the running mean is already a good enough estimate. LayerNorm and RMSNorm don't have
this problem: they do **exactly the same computation** at training time and at inference time (Ba et al., 2016).

## 9 · On real deep networks: how far does normalization help?

The demo trained all four methods × depths 10/20/30 × 6 values of $\eta$ × 3 seeds, each for 1 epoch. Before reading
those results, we rerun one row of it: the 10-layer network, seed 1, $\eta = 1$, a large learning rate. In section 6,
the network without normalization learned well with $\eta = 0.03$ and the BatchNorm network with $\eta = 0.3$. For
each run we print the validation accuracy and the fraction of ReLU neurons that are “completely dead” (output 0 for
all of the first 500 validation images).

In [ ]:
def stored(norm, depth, eta, seed=1):
    run = next(r for r in sweep["runs"] if (r["norm"], r["depth"], r["eta"]) == (norm, depth, eta))
    return run["seeds"][sweep["seeds"].index(seed)]


def show(r):
    return (
        f"blows up at step {r['nanStep']}"
        if "nanStep" in r
        else f"{pct(r['val'])} correct, {pct(r['dead'], 1)} dead neurons"
    )


live, nets = {}, {}
for norm in NORMS:
    P, boom = train(norm, 10, 1.0)
    nets[norm] = P
    live[norm] = {"nanStep": boom} if boom else {"val": accuracy(P, Xval, yval), "dead": dead_share(P)}
    print(f"{LABEL[norm]:>16}, η = 1: {show(live[norm]):<36} demo: {show(stored(norm, 10, 1))}")
for norm in ("ln", "rms"):
    guess = np.bincount(forward(nets[norm], Xval, "eval")[2].argmax(1), minlength=10)
    print(
        f"{LABEL[norm]}, η = 1 guesses: {', '.join(f'“{d}” for {num(n, 0)} images' for d, n in enumerate(guess) if n)}"
    )
print(f"validation images that are a 1: {(yval == 1).sum()} / {num(len(yval), 0)}")
fail = "nanStep" in live["none"] or live["none"]["val"] < 0.15  # blew up, or stuck at the level of random guessing
assert fail and abs(live["bn"]["val"] - stored("bn", 10, 1)["val"]) < 0.005, "differs from the demo"
assert abs(val_none - stored("none", 10, 0.03)["val"]) < 5e-4, "differs from the demo"  # section 6's two networks
for norm in ("ln", "rms"):
    assert abs(live[norm]["val"] - stored(norm, 10, 1)["val"]) < 5e-4, "differs from the demo"
    assert abs(live[norm]["dead"] - stored(norm, 10, 1)["dead"]) < 0.01, "differs from the demo"

With the same $\eta = 1$, the four methods react in three very different ways:
- **No normalization:** blows up (at step 182 with seed 1; on a different machine, rounding errors might keep it
  from blowing up and leave it stuck at the level of random guessing instead, which the `fail` check above allows).
- **BatchNorm:** still learns well, and not a single neuron dies.
- **LayerNorm and RMSNorm:** don't blow up, but collapse: about half of the ReLU neurons die completely, and the
  network guesses “1” for every image. Its accuracy is exactly the share of 1s in the validation set.

Why do neurons die with LayerNorm but not with BatchNorm? BatchNorm forces **each neuron** to have mean 0 over the
batch. While $\gamma, \beta$ are still close to 1 and 0, there is always an image that makes it positive, so it
cannot die completely. LayerNorm only forces **each image** to have mean 0 over its neurons. A neuron can still be
negative for every image, and then the ReLU also blocks its gradient.

Why doesn't BatchNorm blow up? Ioffe & Szegedy (2015, section 3.3) point out that BatchNorm **is unchanged when $W$
is multiplied by a number**: $\mathrm{BN}(h \cdot cW) = \mathrm{BN}(hW)$ for every $c > 0$ (ignoring $\epsilon$ and
$b$, and $b$ is always 0 here). The gradient with respect to $W$, on the other hand, becomes $c$ times smaller. The
cell below checks both on section 7's BatchNorm network, then looks at what the large steps did to the magnitude of
$W$ in the $\eta = 1$ run:

In [ ]:
xb, yb = Xtr[:64], ytr[:64]
P3 = {**P_bn, "W": [w * (3 if l == 4 else 1) for l, w in enumerate(P_bn["W"])]}  # multiply W₅ by 3
grads = []
for P in (P_bn, P3):
    cache, h, z = forward(P, xb, "batch")
    grads.append((z, backward(P, cache, h, z, yb)["W"][4]))
(z_a, g_a), (z_b, g_b) = grads
norm_ratio = np.linalg.norm(g_a) / np.linalg.norm(g_b)
print(
    f"multiplying W₅ by 3: the logits change by at most {sci(np.abs(z_a - z_b).max())}; "
    f"‖∂L/∂W₅‖ becomes {num(norm_ratio, 4)} times smaller"
)
assert np.abs(z_a - z_b).max() < 1e-4 and abs(norm_ratio - 3) < 1e-3
P0 = init("bn", 10)
growth = [np.linalg.norm(w) / np.linalg.norm(w0) for w, w0 in zip(nets["bn"]["W"], P0["W"])]
print("BatchNorm, η = 1: ‖W‖ after 1 epoch / at the start, layers 1…10:", " · ".join(num(g, 2) for g in growth[:10]))

The logits barely change (only at the level of rounding error and $\epsilon$), while the gradient becomes exactly 3
times smaller. This explains the $\eta = 1$ run. A large step makes $W$ longer (layer 1 is twice as long after 1
epoch), but the network still computes exactly as before, while the gradient shrinks accordingly. The relative step
$\|\Delta W\| / \|W\|$ therefore shrinks with the square of $W$'s length: BatchNorm **lowers its own effective
learning rate** when the steps are too large (Arora et al., 2019). A network without normalization has no such
mechanism: when $W$ gets longer, the activations and the gradients grow with it, and then it blows up. Bjorck et al.
(2018) measured that the main benefit of BatchNorm is that it allows a larger $\eta$.

### 🤔 Predict first
At 10 layers, BatchNorm does help. At 30 layers, with the best $\eta$ for each method, is BatchNorm better or worse
than no normalization? And LayerNorm?

In [ ]:
def summary(norm, depth, eta):
    """The median of 3 seeds (a run that blew up counts as the worst, as in the demo) and the min–max range of the
    runs that didn't blow up."""
    seeds = next(r for r in sweep["runs"] if (r["norm"], r["depth"], r["eta"]) == (norm, depth, eta))["seeds"]
    ok = [s["val"] for s in seeds if "val" in s]
    med = sorted(s.get("val", -1.0) for s in seeds)[1]
    return {
        "med": med if med >= 0 else None,
        "lo": min(ok, default=None),
        "hi": max(ok, default=None),
        "nan": 3 - len(ok),
    }


def best(norm, depth):
    """The η with the highest median (ties go to the smaller η, as in the demo)."""
    cands = [(e, summary(norm, depth, e)) for e in sweep["etas"]]
    return max((c for c in cands if c[1]["med"] is not None), key=lambda c: c[1]["med"])


for depth in sweep["depths"]:
    print(f"{depth} hidden layers, the best η of each method (median of 3 seeds, min–max range):")
    for norm in NORMS:
        eta, s = best(norm, depth)
        print(f"  {LABEL[norm]:>16}: η = {num_eta(eta):<4} {pct(s['med'])} ({pct(s['lo'])}–{pct(s['hi'])})")
    n = best("none", depth)[1]
    for norm in ("bn", "ln", "rms"):
        b = best(norm, depth)[1]
        gap = (
            "the 3-seed ranges are separate" if b["lo"] > n["hi"] or n["lo"] > b["hi"] else "the 3-seed ranges overlap"
        )
        print(f"  {LABEL[norm]} − no normalization: {num(100 * (b['med'] - n['med']), 2)} percentage points, {gap}")
top_bn30 = max(s["val"] for r in sweep["runs"] if (r["norm"], r["depth"]) == ("bn", 30) for s in r["seeds"])
print(f"best run of the 30-layer BatchNorm network, over every η and seed: {pct(top_bn30)}")

The figure below plots the whole grid, like the demo's second card: validation accuracy against $\eta$ (log scale),
one panel per depth. The points are the median of 3 seeds, the vertical bars the min–max of the runs that didn't
blow up, and a × at the bottom marks an $\eta$ at which at least 2/3 of the seeds blew up. Each method has its own
color, marker shape and dash style; the methods are shifted slightly to either side so they don't hide each other.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10, 3.6), sharey=True)
for ax, depth in zip(axes, sweep["depths"]):
    for k, norm in enumerate(NORMS):
        st, shift = STYLE[norm], 1.09 ** (k - 1.5)  # shift on the log scale
        rows = [(e * shift, summary(norm, depth, e)) for e in sweep["etas"]]
        ok = [(x, s) for x, s in rows if s["med"] is not None]
        ax.plot([x for x, _ in ok], [100 * s["med"] for _, s in ok], color=st["color"], ls=st["ls"], lw=1.5)
        for x, s in ok:
            ax.plot([x, x], [100 * s["lo"], 100 * s["hi"]], color=st["color"], lw=1.2)
        ax.plot(
            [x for x, _ in ok],
            [100 * s["med"] for _, s in ok],
            st["marker"],
            color=st["color"],
            mfc=st["mfc"],
            ms=6,
            label=LABEL[norm],
        )
        ax.plot(
            [x for x, s in rows if s["med"] is None],
            [3] * sum(s["med"] is None for _, s in rows),
            "x",
            color=st["color"],
            ms=8,
            mew=2,
        )
    ax.axhline(10, color=GREY, lw=0.8, ls=":")
    ax.set(xscale="log", title=f"{depth} hidden layers", xlabel="η (log scale)", ylim=(0, 100))
    ax.grid(axis="y", alpha=0.3)
axes[0].set_ylabel("validation accuracy (%)")
axes[0].text(0.012, 12, "random guessing 10%", fontsize=8, color=GREY)
axes[0].legend(frameon=False, fontsize=8, loc="lower left", bbox_to_anchor=(0, 0.12))
fmt_axes(fig)
for ax in axes:
    ax.set_xticks(sweep["etas"], [num_eta(e) for e in sweep["etas"]], fontsize=8)
    ax.minorticks_off()
plt.tight_layout()
plt.show()

At 10 and 20 layers, BatchNorm wins clearly: its 3-seed range lies entirely above that of the network without
normalization. LayerNorm is also clearly better, but by less; RMSNorm is better by the least (at 10 layers, still
within the variation across seeds). At 30 layers, no method is clearly better than no normalization. BatchNorm's
median drops from 93.92% (20 layers) to 65.96%, 11 points below the network without normalization, and its best run
over every $\eta$ and seed only reaches 75.90%. The 3-seed ranges of the two methods still overlap, so that 11-point
gap lies **within the variation across seeds**: at each method's best $\eta$, all we can say is that BatchNorm is
*no better*. The cell below compares them at each $\eta$ at 30 layers:

In [ ]:
print("30 hidden layers: BatchNorm versus no normalization at each η (min–max of 3 seeds, runs that blew up excluded)")
for eta in sweep["etas"]:
    b, n = summary("bn", 30, eta), summary("none", 30, eta)
    rng_b = f"{pct(b['lo'])}–{pct(b['hi'])}"
    rng_n = f"{pct(n['lo'])}–{pct(n['hi'])}" if n["lo"] is not None else "all 3 seeds blew up"
    if n["lo"] is None or b["lo"] > n["hi"]:
        verdict = "BatchNorm clearly better"
    elif b["hi"] < n["lo"]:
        verdict = "BatchNorm clearly worse"
    else:
        verdict = "within the variation across seeds"
    print(f"  η = {num_eta(eta):<4}: BatchNorm {rng_b:<16} no normalization {rng_n:<19} → {verdict}")

With a large $\eta$, BatchNorm still keeps the network from blowing up and is far better. But with a small $\eta$,
where the network without normalization learns best, BatchNorm is clearly worse at $\eta = 0.01$. Why does a
technique that helps a 10-layer network not help a 30-layer one?

## 10 · Why doesn't BatchNorm rescue the 30-layer network?

Yang et al. (2019) prove that in a *plain* BatchNorm network (without skip connections), right at initialization,
the gradient **grows exponentially** as it travels back toward the first layers. Changing the initialization or the
activation function doesn't cure it, because the cause is BatchNorm itself. We measure this on seed 1's first
training batch, as in the demo: $\|\partial L/\partial W_l\|$ for each layer, before any update step.

In [ ]:
idx1 = np.random.default_rng(1).permutation(N_TRAIN)[:BATCH]  # seed 1's first batch
x1, y1 = Xtr[idx1], ytr[idx1]


def layer_grads(norm, depth, x=x1, y=y1):
    """‖∂L/∂W_l‖ of each hidden layer at initialization (BatchNorm uses the statistics of the batch x)."""
    P = init(norm, depth)
    cache, h, z = forward(P, x, "train")
    return np.array([np.linalg.norm(g) for g in backward(P, cache, h, z, y)["W"][:depth]])


print("‖∂L/∂W₁‖ / ‖∂L/∂W_L‖ at initialization (in parentheses: the demo)")
for depth in sweep["depths"]:
    row = []
    for norm in NORMS:
        g = layer_grads(norm, depth)
        ref = sweep["initGrad"][norm][str(depth)]
        assert abs(g[0] / g[-1] - ref) < 0.006 * ref, "differs from the demo: check the data again"
        row.append(f"{LABEL[norm]} {num(g[0] / g[-1], 1)} ({num(ref, 1)})")
    print(f"  {depth} layers: " + " · ".join(row))
prof = {norm: layer_grads(norm, 30) for norm in NORMS}

The figure below plots $\|\partial L/\partial W_l\|$ against layer $l$ of the 30-layer network, on a log scale.
Going from right to left means going back from the output layer to the first layer, exactly the direction of
backpropagation.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.6))
layers = np.arange(1, 31)
for norm in NORMS:
    st = STYLE[norm]
    ax.plot(
        layers,
        prof[norm],
        color=st["color"],
        ls=st["ls"],
        marker=st["marker"],
        mfc=st["mfc"],
        ms=4.5,
        lw=1.5,
        label=LABEL[norm],
    )
ax.set(
    yscale="log", xlabel="hidden layer l (1 = next to the input)", ylabel="‖∂L/∂Wₗ‖ at initialization", xlim=(0.5, 30.5)
)
ax.set_title("30-layer network, seed 1's first batch: the gradient at each layer", fontsize=11)
ax.grid(alpha=0.3, which="both")
ax.legend(frameon=False, fontsize=9, ncol=2)
fmt_axes(fig)
plt.show()

In the networks without normalization, with LayerNorm and with RMSNorm, the gradients of the layers are of the same
order: the first layer's is only a few times larger than the last layer's. In the BatchNorm network, the blue line is
a steep straight line on the log scale, which means the gradient **is multiplied by a fixed factor at each layer** on
the way back. The cell below measures that factor (the slope of the best-fitting straight line through layers 2–30)
and compares it with the prediction of Yang et al. (2019, Theorem 3.10). The theorem is for an infinitely wide
network and for the squared norm of the gradient, so the factor for the gradient norm itself is its square root; for
very large batches it tends to $\sqrt{\pi/(\pi-1)}$.

In [ ]:
def yang_rate(B):
    """The per-layer growth factor of ‖gradient‖² in a ReLU + BatchNorm network with batch B (Yang et al., 2019,
    Theorem 3.10)."""
    c = -1 / (B - 1)
    J1 = (math.sqrt(1 - c * c) + (math.pi - math.acos(c)) * c) / math.pi  # the arc-cosine kernel (Cho & Saul, 2009)
    dJ1 = (math.pi - math.acos(c)) / math.pi
    return (B - 1 + dJ1) / ((B - 3) * (1 - J1))


k = np.arange(2, 31)
factor = {norm: float(np.exp(-np.polyfit(k, np.log(prof[norm][1:]), 1)[0])) for norm in NORMS}
print(
    "factor the gradient is multiplied by at each layer on the way back (layer 30 → 2):",
    " · ".join(f"{LABEL[n]} {num(factor[n], 3)}" for n in NORMS),
)
print(
    f"theory for BatchNorm: batch of 64 → {num(math.sqrt(yang_rate(64)), 3)}; very large batch → √(π/(π − 1)) = ",
    end="",
)
print(f"{num(math.sqrt(math.pi / (math.pi - 1)), 3)}; to the power 29: {num(factor['bn'] ** 29, 0)} times")

The real network (64 wide, a single batch) gives a factor very close to the theory. A factor only slightly above 1,
repeated over 29 layers, becomes hundreds of times.

What does a large gradient at the first layer do? Every layer shares one $\eta$. If $\eta$ is small enough for the
first layer to take moderate steps, the last layers barely move; if $\eta$ is large enough for the last layers, the
very first step wrecks the first layer's weights. The cell below runs **exactly one** SGD step with $\eta = 0.03$ on
the 30-layer network and measures how much each layer changes, then follows the gradient ratio over the first 100
steps:

In [ ]:
for norm in ("none", "bn"):
    P0, rec = init(norm, 30), {}

    def watch(step, P, G, rec=rec, P0=P0):
        if G is not None:
            rec[step] = float(np.linalg.norm(G["W"][0]) / np.linalg.norm(G["W"][29]))  # the gradient before step `step`
        if step == 1:
            rec["move"] = [np.linalg.norm(P["W"][l] - P0["W"][l]) / np.linalg.norm(P0["W"][l]) for l in (0, 14, 29)]
            rec["cos"] = float((P["W"][0] * P0["W"][0]).sum() / np.linalg.norm(P["W"][0]) / np.linalg.norm(P0["W"][0]))
            rec["len"] = float(np.linalg.norm(P["W"][0]) / np.linalg.norm(P0["W"][0]))

    train(norm, 30, 0.03, steps=101, watch=watch)
    m = rec["move"]
    print(
        f"{LABEL[norm]}, after 1 step: ‖ΔW‖/‖W‖ at layers 1 · 15 · 30 = {pct(m[0], 2)} · {pct(m[1], 2)} · {pct(m[2], 2)}; ",
        end="",
    )
    print(
        f"W₁ is {num(rec['len'], 2)} times as long and turned by {num(math.degrees(math.acos(min(1.0, rec['cos']))), 0)}°"
    )
    print(f"   ‖∂L/∂W₁‖ / ‖∂L/∂W₃₀‖ at steps 1 · 2 · 11 · 101: {' · '.join(num(rec[t], 1) for t in (1, 2, 11, 101))}")

In the network without normalization, one step moves each layer only a tiny fraction. In the BatchNorm network, with
the same $\eta = 0.03$, the first step replaces $W_1$ with a completely different matrix: $\|\Delta W_1\|$ is larger
than $W_1$ itself, and $W_1$ becomes twice as long and turns by about 60°. Whatever He initialization prepared for
the first layer is wiped out right at step 1, while layer 30 barely budges. After that, the gradient ratio drops
quickly: as $W_1$ gets longer its gradient gets smaller (the scale invariance of section 9), and Yang et al. (2019)
also find that after one optimization step, the network reaches a relatively stable equilibrium.

The 30-layer BatchNorm network is therefore stuck between two choices. With a small $\eta$, the last layers learn
very slowly, and the first layer still gets scrambled at step 1 (at $\eta = 0.01$ it only reaches 48–51%). With a
larger $\eta$, the scrambling is stronger and the first layers' effective learning rate gets lowered. This is the
mechanism Yang et al. describe; the notebook measures it but doesn't prove it is the only reason. The lesson to take
away: **normalization alone is not enough to train very deep networks.** Section 13 shows Lesson 13's solution.

## 11 · LayerNorm, RMSNorm and the Transformer

LayerNorm normalizes each sample with its own statistics, so an image's result doesn't depend on the batch, and it
does the same computation at training time and at inference time. The cell below checks this on 10,000 real inputs
of layer 5 (section 7's network). It also checks a remark by Zhang & Sennrich (2019): when the mean of each row is 0,
RMSNorm coincides exactly with LayerNorm.

In [ ]:
ln_all = norm_forward(A_all, "ln", one, zero)[0]
ln_one = norm_forward(A_all[:1], "ln", one, zero)[0]  # image 1 on its own
bn_8, bn_64 = norm_forward(A_all[:8], "bn", one, zero)[0], norm_forward(A_all[:64], "bn", one, zero)[0]
print(f"LayerNorm of image 1, alone or among 10,000 images: difference {sci(np.abs(ln_one[0] - ln_all[0]).max())}")
print(
    f"BatchNorm of image 1, in a batch of 8 or of 64 images: differences of up to {num(np.abs(bn_8[0] - bn_64[0]).max(), 2)}"
)
centered = A_all - A_all.mean(1, keepdims=True)  # each row has mean 0
gap = np.abs(norm_forward(centered, "rms", one, zero)[0] - norm_forward(centered, "ln", one, zero)[0]).max()
print(f"rows with mean 0: RMSNorm − LayerNorm = {sci(gap)}")
row_mu = np.abs(A_all.mean(1))
print(f"but the real a has |row mean| from {num(row_mu.min(), 2)} to {num(row_mu.max(), 2)}, so the two methods differ")
assert np.abs(ln_one[0] - ln_all[0]).max() < 1e-12 and gap < 1e-12

LayerNorm of an image is identical whether that image stands alone or sits among 10,000 images, while BatchNorm of
the same image changes substantially when the batch goes from 8 to 64 images. In a Transformer, each **token** is a
row: LayerNorm normalizes each token's vector over its dimensions, without needing any other token or sentence. That
is why it suits sequences of varying length and very small batches. Where it sits in the model has also changed:

- The **original Transformer** (Vaswani et al., 2017) puts LayerNorm *after* each sublayer:
  $\mathrm{LN}(x + \mathrm{Sublayer}(x))$ (post-norm).
- **GPT-2** (Radford et al., 2019) moves LayerNorm *before* each sublayer, $x + \mathrm{Sublayer}(\mathrm{LN}(x))$
  (pre-norm), and adds one LayerNorm at the end.
- **LLaMA** (Touvron et al., 2023, section 2.2) keeps pre-norm and switches to RMSNorm. The open models of Qwen and
  DeepSeek also use RMSNorm (their public configuration files have an `rms_norm_eps` parameter).

Zhang & Sennrich (2019) hypothesize that the important part of LayerNorm is dividing by the magnitude, and that
subtracting the mean isn't needed. On the models they tried, RMSNorm reaches quality comparable to LayerNorm while
cutting the running time by 7–64%. Lesson 17 (Transformer & GPT) comes back to pre-norm inside a Transformer block.

In convolutional networks, BatchNorm is still common: each channel is normalized over the whole batch and every
position in the image (Ioffe & Szegedy, 2015, section 3.2). But ConvNeXt (Liu et al., 2022) showed that a modern
convolutional network that uses LayerNorm instead of BatchNorm still trains well, and even slightly better (81.5%
top-1 on ImageNet).

## 12 · Try changing things

The cells below rerun the same functions with different parameters. You can change the values and run them again.

### Noise after BatchNorm, on three seeds
Section 6 only tried one seed. The cell below also runs seeds 2 and 3 (10-layer network, $\eta = 0.3$), and compares
them with clean BatchNorm and with the network without normalization ($\eta = 0.03$) of the same seed, taken from
the demo's file. Change `LEVEL` to try stronger noise (0.5) or `ETA_SHAKE` to try a different learning rate.

In [ ]:
LEVEL, ETA_SHAKE = 0.3, 0.3
shaken = {1: val_shake} if (LEVEL, ETA_SHAKE) == (0.3, 0.3) else {}  # seed 1 already ran in section 6
for seed in (1, 2, 3):
    if seed not in shaken:
        P, _ = train("bn", 10, ETA_SHAKE, seed=seed, shake=LEVEL)
        shaken[seed] = accuracy(P, Xval, yval)
for seed in (1, 2, 3):
    clean, plain = stored("bn", 10, ETA_SHAKE, seed), stored("none", 10, 0.03, seed)
    print(f"seed {seed}: BatchNorm with noise {pct(shaken[seed])} · clean BatchNorm {show(clean)} · ", end="")
    print(f"no normalization {show(plain)}")
clean3 = [stored("bn", 10, ETA_SHAKE, s)["val"] for s in (1, 2, 3)]
plain3 = [stored("none", 10, 0.03, s)["val"] for s in (1, 2, 3)]
for name, v in [("with noise", list(shaken.values())), ("clean", clean3), ("no normalization", plain3)]:
    print(f"  {name:>16}: {pct(min(v))}–{pct(max(v))} (3 seeds)")
print(f"clean BatchNorm, seed 3, η = 0.1: {show(stored('bn', 10, 0.1, 3))}")

Over 3 seeds, the range of BatchNorm with noise lies entirely within the range of clean BatchNorm: the difference is
**within the variation across seeds**. Clean seed 3 only reaches 87.02%: for that seed, $\eta = 0.3$ is already too
large (same seed, $\eta = 0.1$ reaches 95.96%), and the noise actually gives a better result. Section 6's conclusion
still stands: the network tolerates a drift many times larger than the natural drift. Try `LEVEL = 0.5` to see how
strong the noise has to be before it starts to hurt.

### Batch size and how fast the gradient explodes
The formula of Yang et al. (2019) depends on the number of images $B$ in the batch: the smaller the batch, the faster
the gradient grows through each layer. The cell below measures that factor on the 30-layer BatchNorm network at
initialization, with the first $B$ images in seed 1's order. Change `SIZES_B` to experiment.

In [ ]:
SIZES_B = [8, 16, 64, 256]
order1 = np.random.default_rng(1).permutation(N_TRAIN)
for B in SIZES_B:
    g = layer_grads("bn", 30, Xtr[order1[:B]], ytr[order1[:B]])
    f = float(np.exp(-np.polyfit(k, np.log(g[1:]), 1)[0]))
    print(f"batch of {B:>3} images: the gradient is multiplied by {num(f, 3)} per layer ", end="")
    print(f"(theory {num(math.sqrt(yang_rate(B)), 3)}); ‖∂L/∂W₁‖ / ‖∂L/∂W₃₀‖ = {num(g[0] / g[-1], 0)}")

The measured value is always below the theory (the theory is for an infinitely wide network; this one is 64 wide):
only slightly from 16 images per batch up, but clearly at 8 images (1.302 against 1.410; compounded over 29 layers,
the two differ about 10-fold). The trend matches, though: the smaller the batch, the faster the gradient explodes.
That is one more reason to avoid BatchNorm with very small batches. Wu & He (2018) proposed GroupNorm, which
normalizes over groups of channels within each sample, for exactly that case.

## 13 · Limits, and the bridge to the next lesson

Normalization keeps each layer's scale in order and allows a larger $\eta$. But in a *plain* network, every signal
and every gradient still has to pass through all 30 layers, and with BatchNorm each layer multiplies the gradient by
another factor of about 1.2. Lesson 13 adds **skip connections** (residual connections): each layer only *adds* a
part to $h$ instead of replacing it,

$$h \leftarrow h + \mathrm{ReLU}(\mathrm{BN}(h))\,W$$

(the “pre-activation” form of He et al., 2016; the first 784 → 64 layer is still a plain layer). The gradient
therefore has a direct path from the output layer back to the first layer. The cell below builds that network with
**exactly the $W$ matrices of section 10's plain network** and measures the gradient ratio at initialization again,
on the same first batch of seed 1.

In [ ]:
def residual_grads(depth, bn=True, x=x1, y=y1):
    """‖∂L/∂W_l‖ at initialization of the network with skip connections h ← h + ReLU(N(h))·W (Lesson 13), with the same
    W as the plain network."""
    W = init("none", depth)["W"]

    def N(a):  # BatchNorm (γ = 1, β = 0) or nothing at all
        return norm_forward(a, "bn", one, zero) if bn else (a, None)

    def N_back(dy, c):
        return norm_backward(dy, "bn", c, one)[0] if bn else dy

    y0, c0 = N(x @ W[0])  # layer 1: 784 → 64, plain
    h, cache = np.maximum(0, y0), []
    for l in range(1, depth):
        y_l, c_l = N(h)
        r = np.maximum(0, y_l)
        cache.append((r, y_l, c_l))
        h = h + r @ W[l]  # skip connection
    y_out, c_out = N(h)
    z = np.maximum(0, y_out) @ W[-1]
    dz = softmax(z)
    dz[np.arange(len(y)), y] -= 1
    dh = N_back((dz / len(y) @ W[-1].T) * (y_out > 0), c_out)
    g = [0.0] * depth
    for l in range(depth - 1, 0, -1):
        r, y_l, c_l = cache[l - 1]
        g[l] = np.linalg.norm(r.T @ dh)
        dh = dh + N_back((dh @ W[l].T) * (y_l > 0), c_l)  # the gradient goes through the shortcut and through the layer
    g[0] = np.linalg.norm(x.T @ N_back(dh * (y0 > 0), c0))
    return np.array(g)


ratio = {}
for depth in sweep["depths"]:
    plain, res = layer_grads("bn", depth), residual_grads(depth)
    ratio[depth] = (plain[0] / plain[-1], res[0] / res[-1])
    print(f"{depth} layers, BatchNorm: ‖∂L/∂W₁‖ / ‖∂L/∂W_L‖ = {num(ratio[depth][0], 1)} (plain) → ", end="")
    print(f"{num(ratio[depth][1], 1)} (with skip connections)")
print(f"from 10 to 30 layers, the ratio grows by a factor of {num(ratio[30][0] / ratio[10][0], 0)} (plain), ", end="")
print(f"{num(ratio[30][1] / ratio[10][1], 1)} (with skip connections)")
assert abs(ratio[30][1] - 38.56) < 0.05  # exactly as in reel/mnist_mlp/resmlp.py, Lesson 13's data

At 10 layers, the two networks are on par. But going from 10 to 30 layers, the plain network's ratio grows by a
factor of 57 (exponentially), while that of the network with skip connections only grows by a factor of about 2. The
same $W$ matrices, the same BatchNorm; the only difference is the addition $h + \dots$. Lesson 13 will train this
network up to 100 layers and see whether it really learns better.

The next lesson (Lesson 11) moves on to images: convolutional networks. BatchNorm comes back there in the form of one
pair $\mu, \sigma$ per **channel**, computed over the whole batch and every position in the image.

## Summary
- The three normalizations share the formula $\hat x = (x - \mu)/\sqrt{\sigma^2 + \epsilon}$,
  $y = \gamma\hat x + \beta$, and differ only in the axis: BatchNorm by column (each neuron over the batch), LayerNorm
  by row (each sample over the neurons), RMSNorm by row but without subtracting $\mu$ and without $\beta$. The
  backward pass of all three matches finite differences to about $10^{-10}$.
- Without normalization, the input to layer 5 drifts within 1 epoch (the standard deviation grows several-fold);
  BatchNorm keeps its output at a mean near 0 and a standard deviation near 1. But deliberately adding strong noise
  after BatchNorm makes the network worse only within the variation across seeds: blocking drift is not the whole
  reason BatchNorm helps (Santurkar et al., 2018).
- With BatchNorm, $\gamma = \sqrt{\sigma^2+\epsilon}$, $\beta = \mu$ gives back exactly $a$; the bias $b$ before it
  never receives a gradient.
- Inference must use the running mean: the network gets 95.32% right, while with the batch's statistics it gets
  93.88% (batch of 64), 75.47% (batch of 8) and 10.32% (batch of 1, every image gets the same answer).
- BatchNorm is unchanged when $W$ is multiplied by a number, while the gradient becomes exactly that many times
  smaller, so it brakes steps that are too large on its own: with $\eta = 1$ the 10-layer network still reaches
  96.04%, while the network without normalization blows up and LayerNorm and RMSNorm collapse to random guessing.
- At 10 and 20 layers BatchNorm is clearly better; at 30 layers no method is clearly better than no normalization. In
  a plain BatchNorm network the gradient grows by a factor of about 1.21 per layer on the way back (Yang et al.,
  2019): ‖∂L/∂W₁‖ is 899 times ‖∂L/∂W₃₀‖, and the first SGD step replaces $W_1$ with a completely different matrix.
  Skip connections (Lesson 13) bring that ratio down to 38.6.

## Review questions
1. For an activation matrix $a$ (rows = images, columns = neurons), along which axis do BatchNorm and LayerNorm
   compute $\mu, \sigma^2$? Which one makes an image's result depend on the other images?
2. Why does the bias $b$ of the Linear layer right before BatchNorm never receive a gradient? What does its job
   instead?
3. A BatchNorm network gets 95.32% right with proper inference. Why does passing the images through it one at a time,
   still using the batch's statistics, leave only 10.32% correct, and why exactly that number?
4. What does section 6's “BatchNorm with noise” experiment say about the internal covariate shift hypothesis?
5. If you multiply the $W$ of a BatchNorm layer by 3, how do the output and the gradient with respect to $W$ change?
   How does that help when $\eta$ is large?
6. Why does BatchNorm help a 10-layer network but not a plain 30-layer one? What do skip connections change?

<details><summary>Hints and answers</summary>

1. BatchNorm by column (each neuron, over the images of the batch), LayerNorm by row (each image, over its neurons).
   BatchNorm makes an image's result depend on the images in the same batch; LayerNorm doesn't, and it computes the
   same way at training time and at inference time.
2. BatchNorm subtracts the column's mean, so adding a constant $b$ to the whole column changes nothing; in the
   backward pass, $\partial L/\partial a$ summed over a column is always 0. $\beta$ does $b$'s job instead.
3. With one image, the $\mu$ of each neuron is that neuron's own value, so $x - \mu = 0$: the image vanishes right at
   the first layer, every image gives the same logits, and the network always guesses the same digit (a 2). The
   accuracy equals the share of 2s in the test set, 1,032 / 10,000.
4. The noise changes the mean and the scale of every neuron at every step, many times more than the natural drift,
   and yet the network learns almost as well as clean BatchNorm. So blocking drift cannot be the whole reason
   BatchNorm helps.
5. The output barely changes, while the gradient becomes exactly 3 times smaller. Large steps make $W$ longer, and
   when $W$ is long, the relative step $\|\Delta W\|/\|W\|$ gets smaller: BatchNorm lowers its own effective learning
   rate, so it doesn't blow up.
6. In a plain network, BatchNorm multiplies the gradient by another factor of about 1.2 per layer on the way back, so
   at 30 layers the gradient of $W_1$ is 899 times that of $W_{30}$, and the very first step already scrambles the
   first layer. Skip connections give the gradient a direct path back to the first layer; with the same $W$, the
   ratio drops to 38.6.
</details>

## Further reading
- S. Ioffe & C. Szegedy (2015), [Batch Normalization: Accelerating Deep Network Training by Reducing Internal Covariate
  Shift](https://arxiv.org/abs/1502.03167): sections 3.1 (inference with the training set's statistics), 3.2
  (dropping $b$; convolutional networks), 3.3 (scale invariance).
- J. L. Ba, J. R. Kiros & G. E. Hinton (2016), [Layer Normalization](https://arxiv.org/abs/1607.06450).
- B. Zhang & R. Sennrich (2019), [Root Mean Square Layer Normalization](https://arxiv.org/abs/1910.07467).
- S. Santurkar, D. Tsipras, A. Ilyas & A. Madry (2018), [How Does Batch Normalization Help
  Optimization?](https://arxiv.org/abs/1805.11604): the “BatchNorm with noise” experiment and the smoother loss
  landscape.
- J. Bjorck, C. Gomes, B. Selman & K. Q. Weinberger (2018), [Understanding Batch
  Normalization](https://arxiv.org/abs/1806.02375): BatchNorm helps mainly because it allows a large $\eta$.
- S. Arora, Z. Li & K. Lyu (2019), [Theoretical Analysis of Auto Rate-Tuning by Batch
  Normalization](https://arxiv.org/abs/1812.03981).
- G. Yang, J. Pennington, V. Rao, J. Sohl-Dickstein & S. S. Schoenholz (2019), [A Mean Field Theory of Batch
  Normalization](https://arxiv.org/abs/1902.08129): Theorem 3.10, exploding gradients in plain BatchNorm networks.
- Y. Wu & J. Johnson (2021), [Rethinking “Batch” in BatchNorm](https://arxiv.org/abs/2105.07576): PreciseBN and the
  pitfalls of BatchNorm.
- Y. Wu & K. He (2018), [Group Normalization](https://arxiv.org/abs/1803.08494).
- Z. Liu et al. (2022), [A ConvNet for the 2020s](https://arxiv.org/abs/2201.03545), section 2.6: replacing BatchNorm
  with LayerNorm in ConvNeXt.
- H. Touvron et al. (2023), [LLaMA](https://arxiv.org/abs/2302.13971), section 2.2: pre-norm with RMSNorm.
- A. Karpathy, [Neural Networks: Zero to Hero](https://karpathy.ai/zero-to-hero.html), the lecture “Building makemore
  Part 3”: BatchNorm written by hand in PyTorch, with plots of the activations and gradients.

**Next lesson:** [Lesson 11 · Convolutional networks (CNNs)](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/cnn/):
replace the fully connected layer with small filters that slide across the image, so that the network reuses the
same weights at every position.

In [ ]:
import platform

import matplotlib

print(f"Python {platform.python_version()} · numpy {np.__version__} · matplotlib {matplotlib.__version__}")